In [ ]:
!pip install ccxt

import ccxt
import pandas as pd
import time

# 1. Configurações Iniciais
# Usamos os pares de moedas corretos para cada corretora
configuracoes = {
    'binance': 'BTC/USDT',
    'kraken': 'BTC/USDT',
    'coinbase': 'BTC/USD'
}

timeframe = '15m'       # Dados de 15 em 15 minutos
registos_alvo = 105000  # Vamos extrair 105k para garantir que cumprimos os >100k com margem de segurança

print("🚀 A iniciar o Robô de Extração de Dados...\n")

# 2. O Ciclo Principal de Extração
for exchange_id, symbol in configuracoes.items():
    print(f"--- A extrair dados da {exchange_id.upper()} ---")
    
    # Inicializar a corretora
    exchange = getattr(ccxt, exchange_id)()
    
    # Definir a data de início (1 de Janeiro de 2021) em milissegundos
    since = exchange.parse8601('2021-01-01T00:00:00Z')
    
    todos_dados = [] # Lista vazia para ir guardando as "fatias" de dados
    
    # Enquanto não atingirmos o nosso alvo, continua a puxar dados
    while len(todos_dados) < registos_alvo:
        try:
            # Puxar uma fatia de 1000 registos
            dados = exchange.fetch_ohlcv(symbol, timeframe, since, limit=1000)
            
            # Se a corretora devolver uma lista vazia, paramos o ciclo
            if not dados:
                break 
                
            # Adicionar a fatia à nossa lista principal
            todos_dados.extend(dados)
            
            # Atualizar o 'since' para o último momento extraído + 1 milissegundo
            # (Isto garante que o próximo pedido começa exatamente onde este acabou)
            since = dados[-1][0] + 1
            
            print(f"Progresso {exchange_id}: {len(todos_dados)} / {registos_alvo} registos...")
            
            # Fazer o robô "respirar" para respeitar o limite de velocidade da corretora
            time.sleep(exchange.rateLimit / 1000)
            
        except Exception as e:
            print(f"❌ Ocorreu um erro na {exchange_id}: {e}")
            break # Em caso de erro, sai do ciclo para não bloquear
            
    # 3. Transformar em DataFrame e Guardar
    if todos_dados:
        # Criar a tabela do Pandas
        df = pd.DataFrame(todos_dados, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        
        # Converter o timestamp para uma data fácil de ler
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms').astype('datetime64[us]')
        
        # Guardar na pasta data no formato exigido (Parquet)
        caminho = f"data/{exchange_id}_btc.parquet"
        df.to_parquet(caminho, index=False)
        
        print(f"✅ SUCESSO! {exchange_id.upper()} guardado com {len(df)} registos no ficheiro: {caminho}\n")

print("🎉 Fase de Extração (Camada Bronze) 100% concluída!")

In [2]:
# ==========================================
# 1. INICIALIZAÇÃO DO PYSPARK
# ==========================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Criar a SparkSession com 2GB de memória (como o professor ensina)
spark = SparkSession.builder \
    .appName("Projeto__Anomalias") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print(f"✅ PySpark {spark.version} pronto a arrancar!\n")

# ==========================================
# 2. CAMADA SILVER (Leitura e Limpeza)
# ==========================================
# Vamos ler os ficheiros da Camada Bronze
df_binance = spark.read.parquet("/home/jovyan/work/data/binance_btc.parquet")
df_kraken = spark.read.parquet("/home/jovyan/work/data/kraken_btc.parquet")
df_coinbase = spark.read.parquet("/home/jovyan/work/data/coinbase_btc.parquet")

def limpar_dados_silver(df, nome_exchange):
    """
    Função para normalizar os dados:
    - Fica apenas com as colunas que importam (timestamp, preço de fecho e volume).
    - Renomeia as colunas para sabermos de qual exchange vieram.
    - Remove linhas com valores nulos.
    """
    return df.select(
        col("timestamp").alias("tempo"),
        col("close").cast("double").alias(f"preco_{nome_exchange}"),
        col("volume").cast("double").alias(f"volume_{nome_exchange}")
    ).dropna().dropDuplicates(["tempo"]) # Remove nulos e tempos duplicados

# Aplicar a limpeza a cada corretora
silver_binance = limpar_dados_silver(df_binance, "binance")
silver_kraken = limpar_dados_silver(df_kraken, "kraken")
silver_coinbase = limpar_dados_silver(df_coinbase, "coinbase")

# Vamos espreitar o aspeto dos dados da Binance já limpos!
print("--- Amostra da Camada Silver (Binance) ---")
silver_binance.show(5)
silver_kraken.show(5)
silver_coinbase.show(5)

✅ PySpark 3.5.0 pronto a arrancar!

--- Amostra da Camada Silver (Binance) ---
+-------------------+-------------+--------------+
|              tempo|preco_binance|volume_binance|
+-------------------+-------------+--------------+
|2021-01-01 04:00:00|     29289.14|    378.797473|
|2021-01-04 21:45:00|     31014.27|    679.636042|
|2021-01-10 03:45:00|     40316.64|    550.337139|
|2021-01-10 19:15:00|     38251.04|    620.358366|
|2021-01-12 21:00:00|     34357.33|   1204.207428|
+-------------------+-------------+--------------+
only showing top 5 rows

+-------------------+------------+-------------+
|              tempo|preco_kraken|volume_kraken|
+-------------------+------------+-------------+
|2026-05-08 00:00:00|     80062.5|   0.74326011|
|2026-05-10 16:30:00|     81265.7|   0.13643491|
|2026-05-11 06:45:00|     80843.7|   0.61867859|
|2026-05-06 11:00:00|     82407.5|   7.58768243|
|2026-05-08 07:00:00|     79343.2|   0.25509541|
+-------------------+------------+-----------